# Interacting walkers

One walker on a graph follows the walk Hamiltonian `h`. Several walkers
also interfere with each other, and they can interact. `InteractingWalk`
simulates `N` bosons, fermions or distinguishable particles with

```text
H = Σ_ij h_ij a†_i a_j + (U/2) Σ_i n_i (n_i − 1) + V Σ_{edges ij} n_i n_j
```

that is, hopping along the edges (`h = −γA` by default), an on-site
interaction `U` and an interaction `V` between neighbours. For bosons this
is the Bose–Hubbard model on the graph.

This notebook shows two-particle interference, pairs bound by a repulsive
interaction, light cones of two particles on a chain, and how interacting
particles tell apart graphs that one walker cannot.

In [ ]:
import math

import matplotlib.pyplot as plt
import numpy as np

import metatron_qso as q

## Two-particle interference

Two particles start on the two ends of an edge. A single walker moves
between the ends as `cos² t` and `sin² t`. Distinguishable particles move
independently, so they are found on different sites with probability
`cos⁴ t + sin⁴ t`. Bosons bunch: their amplitude to be found apart is
`cos² t − sin² t`, which vanishes at `t = π/4`. This is the Hong–Ou–Mandel
dip. Fermions are always found apart.

In [ ]:
edge = q.Graph.path(2)
times = np.linspace(0, math.pi / 2, 101)
fig, ax = plt.subplots(figsize=(6, 3.5))
for statistics in ["bosons", "fermions", "distinguishable"]:
    walk = q.InteractingWalk(edge, 2, statistics)
    apart = [walk.index_of(s) for s in ([0, 1], [1, 0]) if statistics == "distinguishable" or s == [0, 1]]
    p = [sum(walk.probabilities([0, 1], t)[k] for k in apart) for t in times]
    ax.plot(times, p, label=statistics)
ax.plot(times, np.cos(times) ** 4 + np.sin(times) ** 4, "k:", label="cos⁴ t + sin⁴ t")
ax.axvline(math.pi / 4, color="grey", lw=0.5)
ax.set_xlabel("t")
ax.set_ylabel("P(found apart)")
ax.legend()
plt.show()

bosons = q.InteractingWalk(edge, 2)
print("bosons apart at t = π/4:", bosons.probabilities([0, 1], math.pi / 4)[bosons.index_of([0, 1])])

## Pairs bound by repulsion

Two bosons on the same site of a ring cost the energy `U`. When `U` is
much larger than the hopping, the pair cannot separate, because the
separated states lie far away in energy and nothing can take up the
difference. The pair stays together and moves slowly as a whole. Winkler
et al. observed such repulsively bound pairs with atoms in an optical
lattice (Nature 441, 853 (2006)).

In [ ]:
ring = q.Graph.cycle(12)
times = np.linspace(0, 10, 101)
fig, ax = plt.subplots(figsize=(6, 3.5))
for u in [0.0, 2.0, 5.0, 10.0]:
    walk = q.InteractingWalk(ring, 2, on_site=u)
    together = [walk.index_of([i, i]) for i in range(12)]
    psi0 = walk.basis_state([0, 0])
    p = [np.sum(np.abs(walk.evolve(psi0, t)[together]) ** 2) for t in times]
    ax.plot(times, p, label=f"U = {u:g}")
ax.set_xlabel("t")
ax.set_ylabel("P(both on one site)")
ax.legend()
plt.show()

## Two particles on a chain

On a chain of 31 sites, the densities `⟨n_i⟩` of two bosons started next
to each other in the middle spread in a light cone. With a strong
interaction a pair started on one site moves as a whole. Its hopping is a
second-order process of strength of order `γ²/U`, so its light cone is much
narrower.

In [ ]:
chain = q.Graph.path(31)
times = np.linspace(0, 8, 81)
fig, axes = plt.subplots(1, 3, figsize=(11, 3.5), sharey=True)
for ax, (u, start, title) in zip(axes, [(0.0, [15, 16], "U = 0, neighbours"),
                                         (0.0, [15, 15], "U = 0, one site"),
                                         (8.0, [15, 15], "U = 8, one site")]):
    walk = q.InteractingWalk(chain, 2, on_site=u)
    psi0 = walk.basis_state(start)
    rho = np.array([walk.densities(walk.evolve(psi0, t)) for t in times])
    ax.imshow(rho, aspect="auto", origin="lower", extent=[0, 30, 0, 8], cmap="viridis")
    ax.set_title(title)
    ax.set_xlabel("site")
axes[0].set_ylabel("t")
plt.show()

Fermions on the same chain never share a site: `⟨n_i²⟩ = ⟨n_i⟩`, so
`⟨n_i n_j⟩ − δ_ij ⟨n_i⟩` vanishes on the diagonal, and they tend to stay
apart. Bosons bunch, and distinguishable particles lie in between. The
numbers below give the share of the correlation within two sites of the
diagonal.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(8, 3.5))
for ax, statistics in zip(axes, ["bosons", "fermions"]):
    walk = q.InteractingWalk(chain, 2, statistics)
    psi = walk.evolve(walk.basis_state([15, 16]), 5.0)
    c = walk.pair_correlations(psi) - np.diag(walk.densities(psi))
    ax.imshow(c, origin="lower", cmap="magma")
    ax.set_title(f"{statistics}: ⟨n_i n_j⟩ − δ_ij ⟨n_i⟩, t = 5")
    ax.set_xlabel("j")
axes[0].set_ylabel("i")
plt.show()

near = np.abs(np.subtract.outer(np.arange(31), np.arange(31))) <= 2
for statistics in ["bosons", "distinguishable", "fermions"]:
    walk = q.InteractingWalk(chain, 2, statistics)
    psi = walk.evolve(walk.basis_state([15, 16]), 5.0)
    c = walk.pair_correlations(psi) - np.diag(walk.densities(psi))
    print(f"{statistics:16} {c[near].sum() / c.sum():.3f}")

## Telling graphs apart

On a strongly regular graph the propagator `exp(−iAt)` of one walker is
a combination of `I`, `A` and `J − I − A` whose coefficients depend only
on the graph's parameters. One walker therefore behaves the same on all
strongly regular graphs with the same parameters. The triangular graph `T(8) = J(8, 2)` and the
three Chang graphs are all strongly regular with parameters
(28, 12, 6, 4), and none of them is isomorphic to another.

`certificate(times)` sorts the transition probabilities between all
configurations. Isomorphic graphs give the same list up to rounding, and
`certificate_tolerance(times)` lies far above that rounding. Gamble et al. (Phys.
Rev. A 81, 052313 (2010)) showed that two free particles do not tell
strongly regular graphs with equal parameters apart, while two interacting
bosons did in all their tests.

In [ ]:
graphs = {"T(8)": q.Graph.johnson(8, 2)}
graphs.update({f"Chang {k}": q.Graph.chang(k) for k in (1, 2, 3)})
for name, g in graphs.items():
    print(f"{name:8} {g.strongly_regular_parameters()}  |Aut| = {g.automorphism_group_order()}")

def certificate(g, u):
    return q.InteractingWalk(g, 2, on_site=u).certificate([1.0])

names = list(graphs)
for u in [0.0, 1.0]:
    certs = {name: certificate(g, u) for name, g in graphs.items()}
    tolerance = q.InteractingWalk(graphs["T(8)"], 2, on_site=u).certificate_tolerance([1.0])
    print(f"\nU = {u:g}: certificate distances (tolerance {tolerance:.1e})")
    for a in range(4):
        for b in range(a + 1, 4):
            d = q.interacting_certificate_distance(certs[names[a]], certs[names[b]])
            print(f"  {names[a]:8} {names[b]:8} {d:.2e}")

Without interaction all distances are at the level of rounding. With
`U = 1` every pair differs by more than 10⁻⁴, millions of times the
tolerance, which shows (up to floating-point error) that no two of the
graphs are isomorphic; the canonical labelling agrees. The Shrikhande
and 4 × 4 rook's graphs, both (16, 6, 2, 2), behave the same way:

In [ ]:
s, r = q.Graph.shrikhande(), q.Graph.rook(4, 4)
for u in [0.0, 1.0]:
    print(u, q.interacting_certificate_distance(certificate(s, u), certificate(r, u)))
print("isomorphic:", s.is_isomorphic(r))